# 📊 Data Visualization Tutorial - Part 05: Missing Data Visualization and Data Preprocessing

## Welcome to Part 5: Handling and Visualizing Missing Data!

In this practical tutorial, we'll explore techniques for **visualizing and handling missing data**, along with important preprocessing steps like binning and discretization.

### 🎯 Learning Objectives:
1. **Missing Data Visualization**: Use missingno library to visualize patterns in missing data
2. **Data Quality Assessment**: Identify and understand data gaps
3. **Binning Techniques**: Convert continuous variables into categorical bins
4. **Feature Engineering**: Create meaningful groups from numerical data
5. **Advanced Joint Plots**: Multi-dimensional visualization with hue

### 📚 What You'll Learn:
- How to visualize missing data patterns
- Understanding different types of missingness
- Creating effective missing data reports
- Binning strategies for numerical variables
- Age groups, risk categories, and other practical bins
- Using hue in joint plots for multi-category analysis

### 📊 Dataset Used:
- **Heart Disease UCI Dataset**: Medical data with potential missing values

### 🛠️ Technologies:
- **missingno**: Specialized library for missing data visualization
- **Seaborn & Matplotlib**: Statistical visualizations
- **Pandas**: Data manipulation and binning
- **NumPy**: Numerical operations

### 💡 Why This Matters:
Real-world data is **never perfect**. Missing data is common and can:
- Bias your analysis
- Reduce statistical power
- Lead to incorrect conclusions

Understanding and visualizing missing data is crucial for **data quality** and **reliable insights**!

---

## 1️⃣ Environment Setup

Let's import all necessary libraries, including the specialized **missingno** library for missing data visualization.

In [ ]:
# Core libraries
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

# Missing data visualization
import missingno as msno

# Display settings
%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette('Set2')

# Pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries loaded successfully!")
print(f"📦 Pandas version: {pd.__version__}")
print(f"📦 NumPy version: {np.__version__}")
print(f"📦 Seaborn version: {sns.__version__}")
print(f"📦 missingno installed: {'✓' if 'msno' in dir() else '✗'}")

## 2️⃣ Loading and Initial Exploration

### The Heart Disease Dataset

We'll use the UCI Heart Disease dataset, which contains medical measurements and is perfect for demonstrating data quality visualization.

**Important Variables:**
- **age**: Patient age (years)
- **sex**: Gender (1=male, 0=female)
- **cp**: Chest pain type (0-3, categorical)
- **trestbps**: Resting blood pressure (mm Hg)
- **chol**: Serum cholesterol (mg/dl)
- **fbs**: Fasting blood sugar > 120 mg/dl
- **thalch**: Maximum heart rate achieved
- **num**: Diagnosis (0=no disease, 1-4=disease severity)

### Data Quality Issues:
Real medical datasets often have:
- **Missing values**: Tests not performed or data not recorded
- **Outliers**: Measurement errors or truly unusual cases
- **Inconsistencies**: Data entry errors

In [ ]:
# Load the dataset
print("📥 Loading Heart Disease dataset...\n")

df = pd.read_csv('/kaggle/input/heart-disease-data/heart_disease_uci.csv', 
                 delimiter=',', 
                 encoding='utf-8')

# Display basic information
print(f"📊 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n📋 Column Names and Types:")
print(df.dtypes)

print(f"\n🔍 First look at the data:")
display(df.head())

print(f"\n📈 Basic Statistics:")
display(df.describe())

## 3️⃣ Missing Data Analysis

### Why Visualize Missing Data?

Before handling missing data, we need to **understand the pattern** of missingness:

**Types of Missingness:**

1. **MCAR (Missing Completely At Random)**
   - Missing values are random
   - No pattern or relationship
   - Example: Equipment malfunction

2. **MAR (Missing At Random)**
   - Missingness depends on observed data
   - Example: Older patients less likely to complete certain tests

3. **MNAR (Missing Not At Random)**
   - Missingness depends on the missing value itself
   - Example: Sickest patients unable to complete tests

### Detection Strategy:
- Check percentage of missing data per column
- Look for patterns (are they correlated?)
- Identify rows most affected
- Decide on appropriate handling strategy

In [ ]:
# Analyze missing data
print("🔍 Missing Data Analysis\n")
print("="*60)

# Count missing values per column
missing_counts = df.isnull().sum()
missing_percentages = (df.isnull().sum() / len(df)) * 100

# Create a summary dataframe
missing_summary = pd.DataFrame({
    'Column': missing_counts.index,
    'Missing_Count': missing_counts.values,
    'Missing_Percentage': missing_percentages.values
}).sort_values('Missing_Percentage', ascending=False)

# Display only columns with missing data
missing_summary_filtered = missing_summary[missing_summary['Missing_Count'] > 0]

if len(missing_summary_filtered) > 0:
    print("Columns with Missing Data:")
    print(missing_summary_filtered.to_string(index=False))
    print(f"\n📊 Total missing values: {missing_counts.sum()}")
    print(f"📊 Rows with any missing data: {df.isnull().any(axis=1).sum()} ({(df.isnull().any(axis=1).sum()/len(df)*100):.1f}%)")
else:
    print("✅ No missing data found in the dataset!")
    print("(This is unusual for real-world data - the dataset may have been pre-cleaned)")
    
print("\n" + "="*60)

## 4️⃣ Visual Missing Data Analysis with missingno

### The missingno Library

**missingno** provides specialized visualizations for missing data patterns:

1. **Matrix Plot**: Shows data completeness
   - White = missing
   - Black/colored = present
   - Sparkline on right shows row completeness

2. **Bar Chart**: Missing data counts
   - Height = number of non-null values
   - Compare across variables quickly

3. **Heatmap**: Correlation of missingness
   - Shows if missing values co-occur
   - Helps identify patterns

4. **Dendrogram**: Hierarchical clustering
   - Groups variables by missingness pattern
   - Identifies co-missing variables

In [ ]:
# Create comprehensive missing data visualizations
print("📊 Creating missing data visualizations...\n")

# Check if there's any missing data to visualize
if df.isnull().sum().sum() > 0:
    # Matrix plot
    plt.figure(figsize=(14, 6))
    msno.matrix(df, figsize=(14, 6), fontsize=12, sparkline=True, color=(0.27, 0.52, 1.0))
    plt.title('Missing Data Matrix - Data Completeness Pattern', fontsize=14, fontweight='bold', pad=20)
    plt.show()
    print("✓ Matrix plot shows where data is present (colored) vs missing (white)\n")
    
    # Bar chart
    plt.figure(figsize=(14, 6))
    msno.bar(df, figsize=(14, 6), fontsize=12, color=(0.2, 0.4, 0.6))
    plt.title('Missing Data Bar Chart - Non-null Counts by Column', fontsize=14, fontweight='bold', pad=20)
    plt.show()
    print("✓ Bar chart shows count of non-null values per column\n")
    
    # Heatmap (if enough missing data)
    if df.isnull().sum().sum() > len(df) * 0.01:  # At least 1% missing
        plt.figure(figsize=(12, 10))
        msno.heatmap(df, figsize=(12, 10), fontsize=11)
        plt.title('Missing Data Correlation Heatmap', fontsize=14, fontweight='bold', pad=20)
        plt.show()
        print("✓ Heatmap shows correlation in missingness patterns\n")
else:
    print("ℹ️ No missing data to visualize.")
    print("Let's create a sample with artificial missing data for demonstration:\n")
    
    # Create a copy with some artificial missing data for demonstration
    df_demo = df.copy()
    
    # Randomly remove some values (5% of data)
    np.random.seed(42)
    for col in df_demo.select_dtypes(include=[np.number]).columns[:5]:
        mask = np.random.random(len(df_demo)) < 0.05
        df_demo.loc[mask, col] = np.nan
    
    print(f"Created demo dataset with {df_demo.isnull().sum().sum()} missing values\n")
    
    # Matrix plot
    plt.figure(figsize=(14, 6))
    msno.matrix(df_demo, figsize=(14, 6), fontsize=12, sparkline=True)
    plt.title('Missing Data Matrix (Demo) - Data Completeness Pattern', fontsize=14, fontweight='bold', pad=20)
    plt.show()
    
    # Bar chart
    plt.figure(figsize=(14, 6))
    msno.bar(df_demo, figsize=(14, 6), fontsize=12)
    plt.title('Missing Data Bar Chart (Demo)', fontsize=14, fontweight='bold', pad=20)
    plt.show()

print("\n💡 Interpretation Tips:")
print("- Matrix: Look for patterns (vertical lines = systematic missing data)")
print("- Bar: Shorter bars indicate more missing data")
print("- Heatmap: Values close to 1 or -1 indicate correlated missingness")

## 5️⃣ Binning Numerical Variables

### What is Binning?

**Binning** (also called discretization) is the process of converting continuous numerical variables into categorical bins or groups.

### Why Bin Data?

**Benefits:**
1. **Interpretability**: "Young", "Middle-aged", "Senior" is clearer than exact ages
2. **Reduce Noise**: Smooths out minor variations
3. **Handle Outliers**: Extreme values don't dominate
4. **Create Risk Categories**: Medical/financial risk groups
5. **Simplify Models**: Some algorithms work better with categories

**Drawbacks:**
1. **Information Loss**: Lose precision
2. **Arbitrary Boundaries**: Where to cut?
3. **Reduced Variability**: Can hide important patterns

### Binning Strategies:

1. **Equal Width**: Divide range into equal intervals
   - Simple but may create unbalanced groups
   - Example: 0-25, 25-50, 50-75, 75-100

2. **Equal Frequency (Quantiles)**: Same number of observations per bin
   - Balanced groups
   - Example: Quartiles (25%, 50%, 75%)

3. **Domain-Specific**: Use expert knowledge
   - Medical: Risk categories based on guidelines
   - Example: BMI categories (underweight, normal, overweight, obese)

4. **Data-Driven**: Use clustering or decision trees
   - Let data determine boundaries
   - More complex but can find natural groups

In [ ]:
# Remove missing values for binning demonstration
df_clean = df.dropna()

print("🔄 Binning Numerical Variables\n")
print("="*60)

# 1. Age Binning - Equal Width
if 'age' in df_clean.columns:
    print("\n1️⃣ Age Groups (Equal Width Bins)")
    print("-" * 40)
    
    # Define age bins
    age_bins = [0, 30, 40, 50, 60, 100]
    age_labels = ['Young (<30)', 'Adult (30-40)', 'Middle-aged (40-50)', 
                  'Senior (50-60)', 'Elderly (60+)']
    
    df_clean['age_group'] = pd.cut(df_clean['age'], 
                                    bins=age_bins, 
                                    labels=age_labels, 
                                    include_lowest=True)
    
    print(f"Age range: {df_clean['age'].min():.0f} - {df_clean['age'].max():.0f} years")
    print(f"\nAge Group Distribution:")
    print(df_clean['age_group'].value_counts().sort_index())

# 2. Cholesterol Binning - Quantiles
if 'chol' in df_clean.columns:
    print("\n\n2️⃣ Cholesterol Categories (Quantile-based Bins)")
    print("-" * 40)
    
    # Create quartile-based bins
    df_clean['chol_category'] = pd.qcut(df_clean['chol'], 
                                        q=4, 
                                        labels=['Low', 'Medium-Low', 'Medium-High', 'High'],
                                        duplicates='drop')
    
    print(f"Cholesterol range: {df_clean['chol'].min():.0f} - {df_clean['chol'].max():.0f} mg/dl")
    print(f"\nCholesterol Category Distribution:")
    print(df_clean['chol_category'].value_counts().sort_index())

# 3. Blood Pressure Binning - Medical Guidelines
if 'trestbps' in df_clean.columns:
    print("\n\n3️⃣ Blood Pressure Categories (Medical Guidelines)")
    print("-" * 40)
    
    # Based on American Heart Association guidelines
    bp_bins = [0, 120, 130, 140, 180, 300]
    bp_labels = ['Normal (<120)', 'Elevated (120-130)', 'Stage 1 (130-140)',
                 'Stage 2 (140-180)', 'Crisis (>180)']
    
    df_clean['bp_category'] = pd.cut(df_clean['trestbps'],
                                     bins=bp_bins,
                                     labels=bp_labels,
                                     include_lowest=True)
    
    print(f"Blood Pressure range: {df_clean['trestbps'].min():.0f} - {df_clean['trestbps'].max():.0f} mm Hg")
    print(f"\nBlood Pressure Category Distribution:")
    print(df_clean['bp_category'].value_counts().sort_index())

print("\n" + "="*60)
print("\n✅ Binning completed! New categorical columns created.")

## 6️⃣ Visualizing Binned Data

Now that we've created categorical bins, let's visualize how they distribute and relate to our target variable (heart disease diagnosis).

In [ ]:
# Create comprehensive visualization of binned data
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Binned Variable Distributions and Relationships', 
             fontsize=16, fontweight='bold', y=0.995)

# 1. Age Group Distribution
if 'age_group' in df_clean.columns:
    ax1 = axes[0, 0]
    age_counts = df_clean['age_group'].value_counts().sort_index()
    colors = plt.cm.viridis(np.linspace(0, 1, len(age_counts)))
    age_counts.plot(kind='bar', ax=ax1, color=colors, edgecolor='black', alpha=0.8)
    ax1.set_title('Age Group Distribution', fontsize=13, fontweight='bold', pad=10)
    ax1.set_xlabel('Age Group', fontsize=11)
    ax1.set_ylabel('Count', fontsize=11)
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for i, v in enumerate(age_counts.values):
        ax1.text(i, v + 5, str(v), ha='center', va='bottom', fontweight='bold')

# 2. Cholesterol Category Distribution
if 'chol_category' in df_clean.columns:
    ax2 = axes[0, 1]
    chol_counts = df_clean['chol_category'].value_counts().sort_index()
    colors = plt.cm.coolwarm(np.linspace(0, 1, len(chol_counts)))
    chol_counts.plot(kind='bar', ax=ax2, color=colors, edgecolor='black', alpha=0.8)
    ax2.set_title('Cholesterol Category Distribution', fontsize=13, fontweight='bold', pad=10)
    ax2.set_xlabel('Cholesterol Category', fontsize=11)
    ax2.set_ylabel('Count', fontsize=11)
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(axis='y', alpha=0.3)
    
    for i, v in enumerate(chol_counts.values):
        ax2.text(i, v + 5, str(v), ha='center', va='bottom', fontweight='bold')

# 3. Blood Pressure Category Distribution
if 'bp_category' in df_clean.columns:
    ax3 = axes[1, 0]
    bp_counts = df_clean['bp_category'].value_counts().sort_index()
    colors = plt.cm.RdYlGn_r(np.linspace(0, 1, len(bp_counts)))
    bp_counts.plot(kind='bar', ax=ax3, color=colors, edgecolor='black', alpha=0.8)
    ax3.set_title('Blood Pressure Category Distribution', fontsize=13, fontweight='bold', pad=10)
    ax3.set_xlabel('BP Category', fontsize=11)
    ax3.set_ylabel('Count', fontsize=11)
    ax3.tick_params(axis='x', rotation=45)
    ax3.grid(axis='y', alpha=0.3)
    
    for i, v in enumerate(bp_counts.values):
        ax3.text(i, v + 5, str(v), ha='center', va='bottom', fontweight='bold')

# 4. Age vs Disease Presence
if 'age_group' in df_clean.columns and 'num' in df_clean.columns:
    ax4 = axes[1, 1]
    # Create a crosstab
    disease_by_age = pd.crosstab(df_clean['age_group'], 
                                  df_clean['num'].apply(lambda x: 'Disease' if x > 0 else 'No Disease'),
                                  normalize='index') * 100
    
    disease_by_age.plot(kind='bar', stacked=True, ax=ax4, 
                       color=['#2ecc71', '#e74c3c'], edgecolor='black', alpha=0.8)
    ax4.set_title('Heart Disease Prevalence by Age Group', fontsize=13, fontweight='bold', pad=10)
    ax4.set_xlabel('Age Group', fontsize=11)
    ax4.set_ylabel('Percentage (%)', fontsize=11)
    ax4.tick_params(axis='x', rotation=45)
    ax4.legend(title='Diagnosis', loc='upper left')
    ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("- Top row: Distribution of patients across different categories")
print("- Bottom left: Blood pressure risk categories")
print("- Bottom right: How disease prevalence changes with age")
print("\n💡 Notice how binning makes patterns more visible and interpretable!")

## 7️⃣ Advanced Joint Plot with Hue

### Multi-Category Joint Plots

We can enhance joint plots by adding a **hue** parameter, which colors points by a categorical variable. This adds a third dimension of information!

**What Hue Shows:**
- Different colors for different categories
- Separate distributions for each group
- Group-specific patterns and relationships

**Use Cases:**
- Compare males vs females
- Disease vs no disease
- Different age groups or risk categories
- Treatment groups in experiments

### Reading Multi-Hue Joint Plots:
1. **Center scatter**: Look for group separation
2. **Marginal distributions**: Compare shapes between groups
3. **Color overlap**: Shows where groups are similar
4. **Color separation**: Shows where groups differ

In [ ]:
# Create joint plot with chest pain type as hue
print("🎨 Creating advanced joint plot with categorical hue\n")

if 'chol' in df_clean.columns and 'trestbps' in df_clean.columns and 'cp' in df_clean.columns:
    # Create the joint plot
    jointplot = sns.jointplot(
        x='chol',  # Cholesterol
        y='trestbps',  # Resting blood pressure
        data=df_clean,
        hue='cp',  # Chest pain type (0-3)
        palette='winter',  # Color palette
        kind='scatter',  # Scatter plot type
        height=10,
        alpha=0.6,
        marginal_kws={'alpha': 0.5}  # Marginal plot settings
    )
    
    # Customize the plot
    jointplot.fig.suptitle('Cholesterol vs Blood Pressure by Chest Pain Type',
                          fontsize=14, fontweight='bold', y=1.02)
    jointplot.set_axis_labels('Cholesterol (mg/dl)', 'Resting Blood Pressure (mm Hg)', 
                             fontsize=12)
    
    plt.show()
    
    print("\n📋 Chest Pain Types (cp):")
    print("  0: Typical angina")
    print("  1: Atypical angina")
    print("  2: Non-anginal pain")
    print("  3: Asymptomatic")
    
    print("\n🔍 What to Look For:")
    print("- Different chest pain types shown in different colors")
    print("- Top histograms: Cholesterol distribution by pain type")
    print("- Right histograms: Blood pressure distribution by pain type")
    print("- Center: Relationship between variables, colored by pain type")
    print("- Clusters or separations indicate different risk profiles")
else:
    print("⚠️ Required columns not available for joint plot")

print("\n💡 Clinical Insight:")
print("This visualization helps identify if certain chest pain types")
print("are associated with different cholesterol or blood pressure patterns.")